# Classification Algorithms in TuiML

This tutorial covers the classification algorithms available in TuiML, including decision trees, Bayesian classifiers, neural networks, SVMs, and ensemble methods.

In [1]:
# Common imports from TuiML
import numpy as np
from tuiml.evaluation import (
    train_test_split,
    StratifiedKFold,
    accuracy_score,
    classification_report,
    confusion_matrix
)
from tuiml.preprocessing import StandardScaler
from tuiml.datasets import load_iris, load_diabetes, load_glass

## 1. Decision Trees

TuiML provides several tree-based classifiers.

In [2]:
from tuiml.algorithms.trees import C45TreeClassifier, RandomForestClassifier, DecisionStumpClassifier, ReducedErrorPruningTreeClassifier

# Load and prepare data
X, y = load_iris()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# C45TreeClassifier (C4.5 Decision Tree, formerly J48)
j48 = C45TreeClassifier(confidence_factor=0.25)
j48.fit(X_train, y_train)
print(f"C45TreeClassifier Accuracy: {accuracy_score(y_test, j48.predict(X_test)):.2%}")

# Random Forest
rf = RandomForestClassifier(n_estimators=10, random_state=42)
rf.fit(X_train, y_train)
print(f"RandomForestClassifier Accuracy: {accuracy_score(y_test, rf.predict(X_test)):.2%}")

# ReducedErrorPruningTreeClassifier (REPTree)
rep = ReducedErrorPruningTreeClassifier()
rep.fit(X_train, y_train)
print(f"ReducedErrorPruningTreeClassifier Accuracy: {accuracy_score(y_test, rep.predict(X_test)):.2%}")

C45TreeClassifier Accuracy: 100.00%
RandomForestClassifier Accuracy: 100.00%
ReducedErrorPruningTreeClassifier Accuracy: 96.67%


## 2. Bayesian Classifiers

Probabilistic classifiers based on Bayes' theorem.

In [3]:
from tuiml.algorithms.bayesian import NaiveBayesClassifier, BayesianNetworkClassifier

# Naive Bayes
nb = NaiveBayesClassifier()
nb.fit(X_train, y_train)
print(f"NaiveBayesClassifier Accuracy: {accuracy_score(y_test, nb.predict(X_test)):.2%}")

# Get probability predictions
proba = nb.predict_proba(X_test[:5])
print(f"\nProbability predictions (first 5 samples):")
print(proba)

NaiveBayesClassifier Accuracy: 100.00%

Probability predictions (first 5 samples):
[[3.45338103e-089 9.95630462e-001 4.36953775e-003]
 [1.00000000e+000 5.15591174e-014 6.81404875e-021]
 [1.84986846e-286 4.92345956e-012 1.00000000e+000]
 [5.12514708e-093 9.77566816e-001 2.24331843e-002]
 [1.39284148e-104 8.69884553e-001 1.30115447e-001]]


## 3. K-Nearest Neighbors

In [4]:
from tuiml.algorithms.neighbors import KNearestNeighborsClassifier

# KNearestNeighborsClassifier (Instance-Based k-NN, formerly IBk)
knn = KNearestNeighborsClassifier(k=5)
knn.fit(X_train, y_train)
print(f"KNearestNeighborsClassifier (k=5) Accuracy: {accuracy_score(y_test, knn.predict(X_test)):.2%}")

# Try different k values with cross-validation
print("\nCross-validation with different k values:")
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for k in [1, 3, 5, 7, 9]:
    scores = []
    for train_idx, test_idx in kfold.split(X, y):
        knn = KNearestNeighborsClassifier(k=k)
        knn.fit(X[train_idx], y[train_idx])
        scores.append(accuracy_score(y[test_idx], knn.predict(X[test_idx])))
    scores = np.array(scores)
    print(f"k={k}: {scores.mean():.2%} (+/- {scores.std() * 2:.2%})")

KNearestNeighborsClassifier (k=5) Accuracy: 100.00%

Cross-validation with different k values:
k=1: 95.33% (+/- 3.27%)
k=3: 96.67% (+/- 4.22%)
k=5: 96.67% (+/- 4.22%)
k=7: 96.67% (+/- 4.22%)
k=9: 97.33% (+/- 4.99%)


## 4. Support Vector Machines

In [5]:
from tuiml.algorithms.svm import SVC

# Scale features for SVM
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# SVC (Support Vector Classifier, formerly SMO)
svc = SVC(kernel='rbf', C=1.0)
svc.fit(X_train_scaled, y_train)
print(f"SVC (RBF kernel) Accuracy: {accuracy_score(y_test, svc.predict(X_test_scaled)):.2%}")

SVC (RBF kernel) Accuracy: 100.00%


## 5. Neural Networks

In [6]:
from tuiml.algorithms.neural import MultilayerPerceptronClassifier, PerceptronClassifier

# Multilayer Perceptron
mlp = MultilayerPerceptronClassifier(hidden_layers=[10, 5], learning_rate=0.1)
mlp.fit(X_train_scaled, y_train)
print(f"MultilayerPerceptronClassifier Accuracy: {accuracy_score(y_test, mlp.predict(X_test_scaled)):.2%}")

# Simple Perceptron (linear classifier)
perceptron = PerceptronClassifier(learning_rate=0.1)
perceptron.fit(X_train_scaled, y_train)
print(f"PerceptronClassifier Accuracy: {accuracy_score(y_test, perceptron.predict(X_test_scaled)):.2%}")

MultilayerPerceptronClassifier Accuracy: 93.33%
PerceptronClassifier Accuracy: 100.00%


## 6. Ensemble Methods

Combine multiple models for better performance.

In [7]:
from tuiml.algorithms.ensemble import AdaBoostClassifier, BaggingClassifier, VotingClassifier
from tuiml.algorithms.trees import DecisionStumpClassifier

# AdaBoost with Decision Stumps
ada = AdaBoostClassifier(base_classifier=DecisionStumpClassifier(), n_estimators=50)
ada.fit(X_train, y_train)
print(f"AdaBoostClassifier Accuracy: {accuracy_score(y_test, ada.predict(X_test)):.2%}")

# Bagging with C45TreeClassifier
bagging = BaggingClassifier(base_classifier=C45TreeClassifier(), n_estimators=10)
bagging.fit(X_train, y_train)
print(f"BaggingClassifier Accuracy: {accuracy_score(y_test, bagging.predict(X_test)):.2%}")

AdaBoostClassifier Accuracy: 100.00%
BaggingClassifier Accuracy: 100.00%


## 7. Comparing Multiple Classifiers

In [8]:
# Compare classifiers on the Glass dataset
X, y = load_glass()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

classifiers = [
    ('C45TreeClassifier', C45TreeClassifier()),
    ('RandomForestClassifier', RandomForestClassifier(n_estimators=10)),
    ('NaiveBayesClassifier', NaiveBayesClassifier()),
    ('KNearestNeighborsClassifier (k=5)', KNearestNeighborsClassifier(k=5)),
    ('AdaBoostClassifier', AdaBoostClassifier(n_estimators=50)),
]

print("Classifier Comparison (5-fold CV on Glass dataset):")
print("=" * 50)

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, clf in classifiers:
    scores = []
    for train_idx, test_idx in kfold.split(X_scaled, y):
        clf.fit(X_scaled[train_idx], y[train_idx])
        scores.append(accuracy_score(y[test_idx], clf.predict(X_scaled[test_idx])))
    scores = np.array(scores)
    print(f"{name:<35}: {scores.mean():.2%} (+/- {scores.std() * 2:.2%})")

Classifier Comparison (5-fold CV on Glass dataset):
C45TreeClassifier          : 35.55% (+/- 2.26%)
RandomForestClassifier             : 79.01% (+/- 11.10%)
NaiveBayesClassifier               : 43.62% (+/- 16.49%)
KNearestNeighborsClassifier (k=5)  : 69.53% (+/- 18.80%)
AdaBoostClassifier                 : 56.05% (+/- 12.30%)


## 8. Detailed Evaluation

In [9]:
# Train best model and get detailed metrics
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

rf = RandomForestClassifier(n_estimators=20, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Classification Report:
          class  precision     recall   f1-score    support
              0     0.7500     0.8182     0.7826         11
              1     0.8824     0.7500     0.8108         20
              2     0.4000     1.0000     0.5714          2
              4     1.0000     1.0000     1.0000          3
              5     0.0000     0.0000     0.0000          1
              6     1.0000     1.0000     1.0000          5
------------------------------------------------------------
       accuracy                           0.8095         42
      macro avg     0.6721     0.7614     0.6941         42


Confusion Matrix:
[[ 9  1  1  0  0  0]
 [ 3 15  2  0  0  0]
 [ 0  0  2  0  0  0]
 [ 0  0  0  3  0  0]
 [ 0  1  0  0  0  0]
 [ 0  0  0  0  0  5]]


## 9. Using the Experiment Framework

For systematic comparison across multiple datasets, use TuiML's experiment framework.

In [10]:
from tuiml.evaluation import run_experiment

# Define models
models = {
    'C45TreeClassifier': C45TreeClassifier(),
    'RandomForestClassifier': RandomForestClassifier(n_estimators=10),
    'NaiveBayesClassifier': NaiveBayesClassifier(),
    'KNearestNeighborsClassifier': KNearestNeighborsClassifier(k=5)
}

# Define datasets
X_iris, y_iris = load_iris()
X_glass, y_glass = load_glass()

datasets = {
    'Iris': (X_iris, y_iris),
    'Glass': (X_glass, y_glass)
}

# Run 10-fold cross-validation experiment
results = run_experiment(
    models=models,
    datasets=datasets,
    n_folds=10,
    metrics=['accuracy', 'f1']
)

print("\nExperiment Results:")
print(results.summary())


Experiment Results:
Experiment: Experiment
Validation: cross_validation
Metric: accuracy

Dataset: Iris
--------------------------------------------------
  C45TreeClassifier: 0.9400 ± 0.0696
  RandomForestClassifier: 0.9600 ± 0.0680
  NaiveBayesClassifier: 0.9467 ± 0.0718
  KNearestNeighborsClassifier: 0.9733 ± 0.0442

Dataset: Glass
--------------------------------------------------
  C45TreeClassifier: 0.3557 ± 0.0143
  RandomForestClassifier: 0.7320 ± 0.0725
  NaiveBayesClassifier: 0.4614 ± 0.0977
  KNearestNeighborsClassifier: 0.7111 ± 0.0943

